# Chapter 24: Research & Cutting-Edge Techniques
## Notebook 03 — Mixture of Experts

**Mixture-of-experts** scales model capacity without scaling compute per token: a **gate** routes each token to its **top-k experts**. We implement the router and the sparse forward pass.

### What you'll learn

| Topic | Section |
|-------|--------|
| The gating network | §1 |
| Top-k routing and renormalization | §2 |
| Sparse compute and why MoE scales | §3 |

**Time estimate:** 3 hours

### Key concepts

- **Gate** — a small network producing a distribution over experts.
- **Top-k** — activate only the best k experts per token (sparsity).
- **Renormalization** — gate weights over the selected experts sum to one.
- **Conditional compute** — capacity grows with experts, cost grows with k.

---
*Generated by Berta AI*

## 1-2. The router

The gate scores experts per token; we keep the top-k and renormalize their weights into a distribution.

In [ ]:
import sys
sys.path.insert(0, "../scripts")
import numpy as np
from moe import top_k_gating

rng = np.random.default_rng(24)
x = rng.standard_normal((3, 16))      # 3 tokens, d=16
W_gate = rng.standard_normal((16, 4)) # 4 experts
idx, weights = top_k_gating(x, W_gate, k=2)
print("selected experts per token:\n", idx)
print("gate weights sum to 1:", np.allclose(weights.sum(axis=1), 1.0))

## 3. Sparse forward pass

Each token's output is the gate-weighted sum of just its selected experts — full capacity, fractional cost.

In [ ]:
from moe import moe_forward
experts = [lambda x, s=s: x * s for s in (1.0, 2.0, 3.0, 4.0)]  # toy experts
y, idx, weights = moe_forward(x, W_gate, experts, k=2)
print("output shape:", y.shape)

---

## Summary

Mixture-of-experts routes each token to its top-k experts via a gate and combines them with renormalized weights — decoupling model capacity from per-token compute, the trick behind many frontier LLMs.

---
*Generated by Berta AI | Berta Chapters*